# Patient-grouped nested cross-validation — transformer text models

**Revised methodology-aligned implementation (not yet executed).** This replaces the former single train/validation/test split. Five experiments: BERT and BioClinicalBERT each on prognosis-only and combined English text; Danish BERT on original Danish prognosis text. Outer and inner `StratifiedGroupKFold` (5 folds each), patient-disjoint. Class-weighted focal loss; AdamW, learning rate 1e-5, weight decay 0.01. Inner CV selects epoch count and minority-class threshold using **inner out-of-fold predictions**, with no outer-label use. Outer test predictions are pooled for ROC and patient-level bootstrap confidence intervals.

**Important:** This is a *new analysis pipeline*. Results cannot be claimed to reproduce the manuscript until executed and checked against original results. Inspect `PATIENT_COL`, `LABEL_COL`, `ALL_TEXT_COLS`, class encoding, and original model checkpoint names before running. Patient data are never exported to GitHub. Computational cost is substantial (5 models × 5 outer folds × 5 inner folds × up to 5 epochs, plus outer refits).

In [ ]:
import os, json, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support, precision_recall_curve, roc_curve
import matplotlib.pyplot as plt
os.environ["TOKENIZERS_PARALLELISM"] = "false"
SEED=42
DATA_PATH=Path("data/Erda, final, anonymized,for LLM and multimodal.csv")
RESULTS_DIR=Path("results_nested_cv"); RESULTS_DIR.mkdir(exist_ok=True)
PATIENT_COL="patient_id" # REQUIRED: change to exact anonymized patient identifier column
LABEL_COL="is_effective_romexis" # VERIFY encoding: 1 must represent INEFFECTIVE treatment
TEXT_COL_EN="prognosis_english"
TEXT_COL_DA="prognosis_danish"
ALL_TEXT_COLS=["prognosis_english", "subjective_english", "objective_intraoral_english", "objective_extraoral_english"] # REQUIRED: update to actual column names
N_OUTER=5; N_INNER=5; MAX_EPOCHS=5; BOOTSTRAPS=2000
BATCH_SIZE=16
EXPERIMENTS=[
 {"name":"bert_prognosis","checkpoint":"bert-base-uncased","columns":[TEXT_COL_EN],"max_length":128},
 {"name":"bioclinicalbert_prognosis","checkpoint":"emilyalsentzer/Bio_ClinicalBERT","columns":[TEXT_COL_EN],"max_length":128},
 {"name":"bert_all_texts","checkpoint":"bert-base-uncased","columns":ALL_TEXT_COLS,"max_length":128},
 {"name":"bioclinicalbert_all_texts","checkpoint":"emilyalsentzer/Bio_ClinicalBERT","columns":ALL_TEXT_COLS,"max_length":128},
 {"name":"danishbert_prognosis","checkpoint":"Maltehb/danish-bert-botxo","columns":[TEXT_COL_DA],"max_length":128},
]
def seed_all(seed):
 random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
 if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_all(SEED)
print("Device:","cuda" if torch.cuda.is_available() else "cpu")

## Data integrity checks

The binary outcome must be mapped to `1 = ineffective` and `0 = effective`. If the source coding is the opposite, explicitly invert it below **before** running. All required text fields and patient IDs must be present. Missing text is represented as empty text, rather than dropping rows based only on English notes.

In [ ]:
df=pd.read_csv(DATA_PATH)
required={PATIENT_COL,LABEL_COL} | {c for e in EXPERIMENTS for c in e["columns"]}
missing=sorted(required-set(df.columns))
if missing: raise ValueError(f"Missing columns: {missing}. Edit configuration before running.")
if df[PATIENT_COL].isna().any(): raise ValueError("Missing patient identifiers: cannot ensure grouped CV.")
if df[LABEL_COL].isna().any(): raise ValueError("Missing outcome labels: define exclusion criteria explicitly.")
if set(df[LABEL_COL].unique())!={0,1}: raise ValueError("Outcome must be binary 0/1.")
# IMPORTANT: Uncomment only if source data encode 1=effective and 0=ineffective:
# df[LABEL_COL] = 1 - df[LABEL_COL].astype(int)
# Confirm outcome orientation against the study data dictionary before proceeding.
assert df[LABEL_COL].isin([0,1]).all()
for col in required-{PATIENT_COL,LABEL_COL}:
 df[col]=df[col].fillna("").astype(str).str.strip()
print("Cases:",len(df),"Patients:",df[PATIENT_COL].nunique(),"Outcome counts:",df[LABEL_COL].value_counts().to_dict())
print("VERIFY: label 1 = ineffective treatment before training")

## Training and evaluation helpers

In [ ]:
class FocalLoss(nn.Module):
 def __init__(self, alpha, gamma=2.0):
  super().__init__(); self.alpha=alpha; self.gamma=gamma
 def forward(self,logits,targets):
  ce=F.cross_entropy(logits,targets.long(),reduction="none")
  pt=torch.exp(-ce)
  weights=self.alpha.to(logits.device)[targets.long()]
  return (weights*((1-pt)**self.gamma)*ce).mean()

class FocalTrainer(Trainer):
 def __init__(self,*args,focal_alpha=None,**kwargs):
  super().__init__(*args,**kwargs); self.focal_loss=FocalLoss(focal_alpha)
 def compute_loss(self,model,inputs,return_outputs=False,**kwargs):
  inputs=dict(inputs); labels=inputs.pop("labels")
  outputs=model(**inputs); loss=self.focal_loss(outputs.logits,labels)
  return (loss,outputs) if return_outputs else loss

def make_text(frame,cols):
 return frame[cols].fillna("").astype(str).agg(" ".join,axis=1).str.strip().tolist()

def token_dataset(frame,cols,tok,max_length):
 texts=make_text(frame,cols)
 enc=tok(texts,truncation=True,max_length=max_length,padding=False)
 enc["labels"]=frame[LABEL_COL].astype(int).tolist()
 return Dataset.from_dict(enc)

def weights_from_train(y):
 counts=np.bincount(np.asarray(y,dtype=int),minlength=2)
 if (counts==0).any(): raise ValueError("A training fold lacks a class")
 return torch.tensor(len(y)/(2*counts),dtype=torch.float32)

def softmax_class1(logits):
 logits=np.asarray(logits); logits=logits-logits.max(axis=1,keepdims=True)
 ex=np.exp(logits); return ex[:,1]/ex.sum(axis=1)

def train_predict(train_df,val_df,exp,epochs,tag):
 seed_all(SEED)
 tok=AutoTokenizer.from_pretrained(exp["checkpoint"])
 train_ds=token_dataset(train_df,exp["columns"],tok,exp["max_length"])
 val_ds=token_dataset(val_df,exp["columns"],tok,exp["max_length"])
 model=AutoModelForSequenceClassification.from_pretrained(exp["checkpoint"],num_labels=2)
 args=TrainingArguments(output_dir=str(RESULTS_DIR/"checkpoints"/tag),learning_rate=1e-5,weight_decay=0.01,
  per_device_train_batch_size=BATCH_SIZE,per_device_eval_batch_size=BATCH_SIZE,num_train_epochs=int(epochs),
  save_strategy="no",eval_strategy="no",report_to="none",seed=SEED,remove_unused_columns=True)
 trainer=FocalTrainer(model=model,args=args,train_dataset=train_ds,data_collator=DataCollatorWithPadding(tok),
  focal_alpha=weights_from_train(train_df[LABEL_COL].to_numpy()))
 trainer.train()
 prob=softmax_class1(trainer.predict(val_ds).predictions)
 del trainer,model,train_ds,val_ds,tok
 gc.collect()
 if torch.cuda.is_available(): torch.cuda.empty_cache()
 return prob

def choose_threshold(y,prob):
 precision,recall,thresholds=precision_recall_curve(y,prob)
 if not len(thresholds): return 0.5
 f1=2*precision[:-1]*recall[:-1]/np.maximum(precision[:-1]+recall[:-1],1e-12)
 return float(thresholds[np.argmax(f1)])

def metrics(y,prob,threshold):
 pred=(np.asarray(prob)>=threshold).astype(int)
 p,r,f,_=precision_recall_fscore_support(y,pred,labels=[0,1],zero_division=0)
 return {"roc_auc":float(roc_auc_score(y,prob)),"pr_auc":float(average_precision_score(y,prob)),
         "macro_f1":float(np.mean(f)),"failure_precision":float(p[1]),"failure_sensitivity":float(r[1])}

def grouped_bootstrap(pred_df,n_boot=2000,seed=42):
 rng=np.random.default_rng(seed)
 groups={p:group.index.to_numpy() for p,group in pred_df.groupby("patient_id",sort=False)}
 patients=np.array(list(groups),dtype=object); values=[]
 for _ in range(n_boot):
  sample=rng.choice(np.arange(len(patients)),size=len(patients),replace=True)
  indices=np.concatenate([groups[patients[j]] for j in sample])
  sub=pred_df.loc[indices]
  if sub.y_true.nunique()<2: continue
  # Threshold is stored per outer fold, not re-optimized using bootstrap samples.
  y=sub.y_true.to_numpy(); prob=sub.y_prob.to_numpy(); pred=(prob>=sub.threshold.to_numpy()).astype(int)
  p,r,f,_=precision_recall_fscore_support(y,pred,labels=[0,1],zero_division=0)
  values.append({"roc_auc":roc_auc_score(y,prob),"pr_auc":average_precision_score(y,prob),
   "macro_f1":np.mean(f),"failure_precision":p[1],"failure_sensitivity":r[1]})
 return {key:[float(np.percentile([v[key] for v in values],q)) for q in [2.5,97.5]] for key in values[0]} if values else {}


## Nested 5 × 5 patient-grouped CV

Inner out-of-fold predictions are generated separately for each candidate epoch count. We select the number of epochs using inner ROC-AUC, then select a failure-class F1 threshold on those same inner out-of-fold predictions. This is a pragmatic tuning design; its threshold is never selected on outer-test labels. **If the original analysis used different hyperparameters or selection criteria, adjust these to match the original study before interpreting results.**

In [ ]:
def run_nested(exp):
 y=df[LABEL_COL].astype(int).to_numpy(); groups=df[PATIENT_COL].to_numpy()
 outer=StratifiedGroupKFold(n_splits=N_OUTER,shuffle=True,random_state=SEED)
 records=[]; fold_summaries=[]
 for outer_i,(tr,te) in enumerate(outer.split(df,y,groups),start=1):
  train=df.iloc[tr].reset_index(drop=True); test=df.iloc[te].reset_index(drop=True)
  assert set(train[PATIENT_COL]).isdisjoint(test[PATIENT_COL])
  inner=StratifiedGroupKFold(n_splits=N_INNER,shuffle=True,random_state=SEED+outer_i)
  candidates=[]
  for epochs in range(1,MAX_EPOCHS+1):
   oof=np.full(len(train),np.nan)
   for inner_i,(itr,iva) in enumerate(inner.split(train,train[LABEL_COL],train[PATIENT_COL]),start=1):
    a=train.iloc[itr]; b=train.iloc[iva]
    assert set(a[PATIENT_COL]).isdisjoint(b[PATIENT_COL])
    oof[iva]=train_predict(a,b,exp,epochs,f"{exp['name']}_o{outer_i}_e{epochs}_i{inner_i}")
   if np.isnan(oof).any(): raise RuntimeError("Incomplete inner OOF predictions")
   candidates.append((roc_auc_score(train[LABEL_COL],oof),epochs,oof))
  _,best_epochs,best_oof=max(candidates,key=lambda z:z[0])
  threshold=choose_threshold(train[LABEL_COL].to_numpy(),best_oof)
  prob=train_predict(train,test,exp,best_epochs,f"{exp['name']}_outer{outer_i}")
  result=metrics(test[LABEL_COL].to_numpy(),prob,threshold)
  fold_summaries.append({"fold":outer_i,"epochs":best_epochs,"threshold":threshold,"n_test":len(test),**result})
  for j in range(len(test)):
   records.append({"row_id":int(te[j]),"patient_id":str(test.iloc[j][PATIENT_COL]),"outer_fold":outer_i,
    "y_true":int(test.iloc[j][LABEL_COL]),"y_prob":float(prob[j]),"threshold":threshold})
  print(exp["name"],"outer fold",outer_i,result)
 pred_df=pd.DataFrame(records).sort_values("row_id").reset_index(drop=True)
 assert len(pred_df)==len(df) and pred_df.row_id.is_unique
 folds=pd.DataFrame(fold_summaries)
 # Patient IDs are retained only in local outputs; NEVER commit these files to GitHub.
 pred_df.to_csv(RESULTS_DIR/f"{exp['name']}_oof_PRIVATE.csv",index=False)
 folds.to_csv(RESULTS_DIR/f"{exp['name']}_fold_metrics.csv",index=False)
 cis=grouped_bootstrap(pred_df,BOOTSTRAPS,SEED)
 aggregate={"experiment":exp["name"],"n":len(pred_df),"patients":pred_df.patient_id.nunique(),
  "fold_mean":folds.select_dtypes(include="number").mean().to_dict(),
  "fold_sd":folds.select_dtypes(include="number").std().to_dict(),"patient_bootstrap_95ci":cis}
 with open(RESULTS_DIR/f"{exp['name']}_summary.json","w") as f: json.dump(aggregate,f,indent=2)
 return pred_df,folds,aggregate


## Execute experiments

Set `RUN_EXPERIMENTS=True` only after confirming all configuration settings and the outcome encoding. This notebook intentionally does not contain fabricated results.

In [ ]:
RUN_EXPERIMENTS=False
all_results={}
if RUN_EXPERIMENTS:
 for exp in EXPERIMENTS:
  all_results[exp["name"]]=run_nested(exp)
else:
 print("Dry mode: review the configuration, then set RUN_EXPERIMENTS=True.")

## Pooled ROC visualization (only after execution)

In [ ]:
if all_results:
 for name,(pred,folds,summary) in all_results.items():
  fpr,tpr,_=roc_curve(pred.y_true,pred.y_prob)
  plt.plot(fpr,tpr,label=f"{name} (AUC={roc_auc_score(pred.y_true,pred.y_prob):.3f})")
 plt.plot([0,1],[0,1],"--",color="gray")
 plt.xlabel("False positive rate");plt.ylabel("True positive rate")
 plt.legend();plt.tight_layout();plt.show()

## Reproducibility and privacy notes

- Run locally with authorized access to the study data. Confirm actual patient identifier, source label orientation, and clinical text column names.
- The original notebook had a 70/10/20 split and different English checkpoints; the new code **does not reproduce those historical results**.
- `results_nested_cv/*_oof_PRIVATE.csv` contains patient IDs: keep local, never publish. Add `data/`, `results_nested_cv/`, `checkpoints/`, and `*.npz` to `.gitignore` as appropriate.
- Five inner folds may be infeasible if a fold has too few failure patients; check per-fold class counts before training.
- Pooled ROC-AUC and mean ± SD of outer-fold metrics are different estimands; report them distinctly.
- Text standardization/translation is assumed already completed in the input CSV; verify it matches the manuscript.
- The code is syntax-checked, but full training and empirical reproduction require the protected dataset and model downloads.